Los modelos usan Adamw

$$w \leftarrow w - \frac{\eta}{\sqrt{\widetilde{s} + \epsilon}} \widetilde{v} - \eta \lambda w$$

El optimizador en Deep Learning es el mecanismo encargado de calcular y aplicar las actualizaciones sobre los pesos del modelo. Si bien todo parte del descenso de gradiente clásico (*Gradient Descent*), con el tiempo se han incorporado extensiones y ajustes —especialmente sobre el algoritmo Adam— que hacen que la optimización sea más rápida, estable y eficaz.

De hecho, existe una modificación clave sobre la fórmula original de Adam que marca una diferencia crítica al entrenar arquitecturas complejas como los LLMs: la forma en que se maneja la **regularización**.

---

### 1. La Función de Coste y la Búsqueda de Pesos

Imaginemos que $w$ representa el conjunto de todos los pesos y parámetros entrenables del modelo. El objetivo del entrenamiento es encontrar el conjunto de valores de $w$ que minimice la función de coste total $J(w)$, la cual se define como el promedio (o suma) de las pérdidas individuales sobre los elementos de un lote (*batch*):

$$J(w) = \frac{1}{B} \sum_{i=1}^B \mathcal{L}(f(x_i; w), y_i)$$

Para modelos sencillos o conjuntos de datos homogéneos, minimizar únicamente esta pérdida empírica puede ser suficiente. Sin embargo, cuando los modelos escalan a millones o miles de millones de parámetros y los datos son altamente diversos, minimizar solo la pérdida directa conduce inevitablemente al sobreajuste (*overfitting*).

---

### 2. Regularización L2: Penalizar Pesos Excesivos

Para evitar que el modelo dependa excesivamente de características espurias o desarrolle pesos desproporcionados, introducimos un término de regularización directamente en la función objetivo:

$$J_{\text{reg}}(w) = J(w) + \frac{\lambda}{2} \Vert{}w\Vert{}_2^2$$

Donde $\Vert{}w\Vert{}_2^2 = \sum w_j^2$ es la norma L2 al cuadrado (la suma de todos los pesos al cuadrado) y $\lambda$ es el hiperparámetro de regularización (un valor típicamente pequeño como $0.01$ o $0.001$).

* **Propósito:** El optimizador ahora busca minimizar tanto el error de predicción como la magnitud de los pesos.
* **Efecto:** Evita que los pesos crezcan sin control y previene que parámetros individuales adquieran una dominancia excesiva sobre la salida del modelo.

---

### 3. El Problema Oculto: L2 Regularization vs. Adam

En el descenso de gradiente estándar (SGD), la regularización L2 es matemáticamente idéntica al **Weight Decay** (decaimiento de peso).

Sin embargo, al combinar la regularización L2 clásica con **Adam**, surge un desajuste: Adam divide la actualización por la raíz de la media cuadrática de los gradientes históricos ($\sqrt{v_t}$). Esto provoca que los pesos con gradientes acumulados muy grandes reciban *menos* regularización de la debida, mientras que los pesos con gradientes dispersos reciban una penalización desproporcionadamente alta.

Este problema motivó el desarrollo de **AdamW**, donde el término de decaimiento de peso se desacopla por completo del cálculo de gradientes y momentos, convirtiéndose en el estándar absoluto para el entrenamiento de Transformers y LLMs.

In [1]:
# ADAM

En el optimizador **Adam**, cada parámetro del modelo se actualiza calculando dos medias ponderadas exponencialmente (EWMA): el **primer momento ($m_t$ o $v_t$)** que corresponde al Momentum, y el **segundo momento ($s_t$)** que rastrea la variabilidad (magnitud cuadrática) de los gradientes.

---

### 1. El término de Momentum ($m_t$ / Inercia y Dirección)

El primer momento actúa como una velocidad acumulada:

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$

* **Qué hace:** En lugar de depender únicamente del gradiente del lote actual ($g_t$), mantiene un promedio ponderado de la dirección de los pasos anteriores.
* **Efecto:** Ayuda al optimizador a seguir avanzando en direcciones consistentes y cancela las oscilaciones erráticas en curvas estrechas del espacio de pérdidas.

---

### 2. El término de Varianza/Escala ($s_t$ / Tasa Adaptativa)

El segundo momento calcula la media ponderada de los gradientes al cuadrado:

$$s_t = \beta_2 s_{t-1} + (1 - \beta_2) g_t^2$$

Aquí reside la intuición clave sobre la estabilidad y el ritmo de aprendizaje:

* **Pesos inestables con gradientes muy variables:**
Si un peso recibe gradientes gigantescos o cambia drásticamente en cada lote, el valor de $s_t$ será grande. Al actualizar el peso, **dividimos entre $\sqrt{s_t}$**, lo que amortigua el paso y evita que el parámetro salte sin control o desestabilice el entrenamiento.
* **Pesos estancados o con gradientes pequeños (mesetas):**
Si un peso casi no cambia o se encuentra en una región plana donde el gradiente es diminuto, $s_t$ será muy pequeño. Al dividir entre un número cercano a cero, el paso efectivo se amplifica, dándole un impulso extra para sacarlo de la meseta.

---

### 3. La regla de actualización combinada

Tras aplicar la corrección de sesgo ($\hat{m}_t$ y $\hat{s}_t$), la actualización de cada peso $w$ se calcula como:

$$w_{t+1} = w_t - \frac{\eta}{\sqrt{\hat{s}_t} + \epsilon} \cdot \hat{m}_t$$

* $\hat{m}_t$ aporta la **dirección e inercia**.
* $\frac{1}{\sqrt{\hat{s}_t} + \epsilon}$ actúa como un **freno o acelerador individual** para cada peso según su historial de variabilidad.

In [3]:
# ADAMW Los modelos de LLMS

La intuición de que **Weight Decay encoge o contrae los pesos restando una fracción de ellos en cada paso** es matemáticamente correcta. Sin embargo, hay una distinción técnica fundamental entre **Adam con Regularización L2** y **AdamW (Decoupled Weight Decay)**.

---

### 1. El gradiente de la regularización L2

Si definimos la penalización L2 sobre los pesos como:

$$\Omega(w) = \frac{\lambda}{2} \Vert{}w\Vert{}_2^2 = \frac{\lambda}{2} \sum w_j^2$$

Al calcular el gradiente respecto a $w$, la derivada de $w^2$ es $2w$:

$$\nabla_w \left(\frac{\lambda}{2} \Vert{}w\Vert{}_2^2\right) = \lambda w$$

El gradiente de la regularización L2 es directamente proporcional al valor actual del propio peso ($w$).

---

### 2. Por qué Adam clásico con L2 "rompe" la contracción

En **SGD**, sumar el gradiente L2 a la función de pérdida da como resultado:

$$w_{t+1} = w_t - \eta \big( \nabla \mathcal{L}(w_t) + \lambda w_t \big) = (1 - \eta \lambda) w_t - \eta \nabla \mathcal{L}(w_t)$$

El término $(1 - \eta \lambda)$ encoge directamente los pesos hacia cero en cada iteración (*Weight Decay*).

En **Adam clásico**, si agregas la penalización L2 a la pérdida, el gradiente total que entra en los momentos $m_t$ y $s_t$ es:

$$g_t = \nabla \mathcal{L}(w_t) + \lambda w_t$$

Al actualizar, el segundo momento $s_t$ (la media de $g_t^2$) divide a toda la expresión:

$$w_{t+1} = w_t - \frac{\eta}{\sqrt{\hat{s}_t} + \epsilon} \hat{m}_t$$

* **El problema:** Si un peso tiene gradientes históricos muy grandes, $\hat{s}_t$ será enorme, haciendo que la penalización $\lambda w_t$ quede dividida por un número gigante. El peso apenas se contrae.
* Si un peso tiene gradientes pequeños, se penaliza desproporcionadamente.

---

### 3. La solución en AdamW: Desacoplar la contracción

**AdamW** (Loshchilov & Hutter, 2017) extrae la regularización fuera de los momentos $m_t$ y $s_t$, y la aplica directamente a la regla de actualización del peso:

$$w_{t+1} = \underbrace{(1 - \eta \lambda) w_t}_{\text{Contracción directa (Weight Decay)}} - \underbrace{\frac{\eta}{\sqrt{\hat{s}_t} + \epsilon} \hat{m}_t}_{\text{Paso Adam puro sobre la pérdida}}$$

De esta forma:

* Cada peso sufre una contracción multiplicativa constante $(1 - \eta \lambda)$ en cada paso.
* Los momentos adaptativos de Adam optimizan únicamente el error de la tarea ($\mathcal{L}$), sin distorsionar la regularización.

In [4]:
# Explicacion de la ecuación

Esta ecuación nos dice exactamente **cómo cambia cada peso ($w$) en cada paso de entrenamiento**.

Se divide en 3 partes muy sencillas:

---

$$w \leftarrow \mathbf{w} - \underbrace{\frac{\eta}{\sqrt{\widetilde{s} + \epsilon}} \widetilde{v}}_{\text{El paso de Adam}} - \underbrace{\eta \lambda \mathbf{w}}_{\text{La contracción (L2 / Weight Decay)}}$$

---

### 1. $\mathbf{w}$ (El punto de partida)

Es el valor actual que tiene el peso antes de la actualización.

---

### 2. El paso inteligente de Adam: $-\frac{\eta}{\sqrt{\widetilde{s} + \epsilon}} \widetilde{v}$

Es la corrección que hace el modelo para aprender del error:

* $\eta$ (Learning Rate): Qué tan grande es el paso general.
* $\widetilde{v}$ (Momentum / Dirección corregida): La inercia acumulada hacia dónde debe moverse el peso.
* $\sqrt{\widetilde{s}}$ (Varianza / Freno adaptativo):
* Si el peso es **muy inestable**, $\widetilde{s}$ es grande $\to$ **frena el paso**.
* Si el peso está **estancado**, $\widetilde{s}$ es pequeño $\to$ **acelera el paso**.



---

### 3. La contracción L2 / Weight Decay: $-\eta \lambda \mathbf{w}$

Es el mecanismo para evitar que los pesos crezcan sin control (*overfitting*):

* $\lambda$ es una fracción muy pequeña (ej. $0.01$).
* Al restar $\eta \lambda \mathbf{w}$, le quitamos **un pequeño porcentaje del propio peso**.
* **Resultado:** Si un peso se vuelve innecesariamente grande, esta resta lo "encoge" (*decays*) hacia cero de forma automática en cada iteración.

> **Nota clave sobre la diferencia:**
> * **En Adam clásico (con L2 tradicional):** Regularizamos la **actualización** (el gradiente), no directamente los pesos. La penalización se mezcla dentro de los momentos adaptativos y termina distorsionada al dividirse entre la escala $\sqrt{\widetilde{s}}$.
> * **En AdamW (Decoupled Weight Decay):** Regularizamos directamente **las propias ponderaciones (los pesos)** en lugar del cálculo de actualización. El término $-\eta\lambda w$ encoge el peso de forma pura e independiente, sin importar la magnitud del gradiente.
>
>